# CS781 Phase 3 — Experiment 1
## Model 4 + LoRA for Rhetorical Role Prediction — Improved

**Backbone:** Phase-1 Model 4 Efficient Causal Decoder (~99.3M)  
**Adaptation:** custom LoRA from scratch on attention (`q,k,v,o`) and MLP (`gate,up,down`) projections; Phase-1 base weights remain frozen  
**Task:** 13-way sentence-level rhetorical-role classification  
**Context:** ±2 neighbouring sentences + target sentence + CL/IT domain marker + relative document-position features  
**Representation:** last-token causal summary + masked mean pooling + position MLP  
**Validation:** exact supplied Phase-3 evaluator; CL + IT scored together  
**Final submission:** refit on train+dev for the best held-out epoch, then test inference and strict `prediction.json` validation

This remains **one LoRA experiment**. The extra context, position features, pooling, and decoding are task-specific improvements and do not change the PEFT identity.

In [1]:
# ============================================================
# CELL 1 — Imports, seed, paths, and experiment config
# ============================================================

import os, re, gc, json, math, time, random, zipfile, subprocess, sys
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from huggingface_hub import HfApi, hf_hub_download, login

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# ----------------------------- Hugging Face auth
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("Could not read Kaggle HF_TOKEN secret:", repr(e))

assert HF_TOKEN, "HF_TOKEN not found in Kaggle Secrets."
login(token=HF_TOKEN)

HF_REPO = "Exploration-Lab/CS-781-Capstone"

# ----------------------------- output
ROOT = Path("/kaggle/working/phase3_exp1_model4_lora")
ROOT.mkdir(parents=True, exist_ok=True)

BEST_CKPT = ROOT / "best_lora_rr.pt"
FINAL_CKPT = ROOT / "final_refit_lora_rr.pt"

DEV_PRED = ROOT / "prediction_dev.json"
CL_DEV_TRUTH = ROOT / "cl_test_truth.json"
IT_DEV_TRUTH = ROOT / "it_test_truth.json"
LABELS_JSON = ROOT / "labels.json"
EVAL_SCRIPT = ROOT / "evaluate_phase3.py"

OFFICIAL_PRED = ROOT / "prediction.json"
SUBMISSION_ZIP = ROOT / "submission.zip"

# ----------------------------- exact Phase-1 Model 4 config
VOCAB_SIZE = 24_576
D_MODEL = 640
N_LAYERS = 17
N_HEADS = 10
N_KV_HEADS = 2
FFN_DIM = 2048
MAX_POSITIONS = 2560
ROPE_BASE = 10_000.0
RMS_EPS = 1e-5
DROPOUT = 0.0

# ----------------------------- RR / LoRA config
MAX_LEN = 640
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
NUM_EPOCHS = 6
EARLY_STOP_PATIENCE = 2
CONTEXT_SENTS = 2          # 0 = sentence-only ablation
USE_POS = True             # False = no position features (ablation)
LORA_TARGETS = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

MICRO_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
LORA_LR = 2e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_RATIO = 0.06
USE_AMP = True

HEAD_DROPOUT = 0.15
CLASS_WEIGHT_MIN = 0.35
CLASS_WEIGHT_MAX = 3.50

# Segment token budgets. Target gets priority.
TARGET_BUDGET = 320
PREV_BUDGET = 120
NEXT_BUDGET = 120

print("Output directory:", ROOT)
print("Model4:", N_LAYERS, "layers | d=", D_MODEL, "|", N_HEADS, "Q /", N_KV_HEADS, "KV")
print("LoRA targets:", LORA_TARGETS, "| r=", LORA_R, "| alpha=", LORA_ALPHA)
print("Context sentences each side:", CONTEXT_SENTS, "| use position:", USE_POS)

Device: cuda
GPU: Tesla T4
Output directory: /kaggle/working/phase3_exp1_model4_lora
Model4: 17 layers | d= 640 | 10 Q / 2 KV
LoRA targets: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'] | r= 16 | alpha= 32
Context sentences each side: 2 | use position: True


In [2]:
# ============================================================
# CELL 2 — Find local Phase-1 checkpoint and tokenizer
# ============================================================

def find_unique_input_file(exact_names=None, contains=None, suffix=None):
    matches = []
    for root, _, files in os.walk("/kaggle/input"):
        for fn in files:
            ok = True
            if exact_names is not None:
                ok = fn in exact_names
            if contains is not None:
                ok = ok and all(x.lower() in fn.lower() for x in contains)
            if suffix is not None:
                ok = ok and fn.lower().endswith(suffix.lower())
            if ok:
                matches.append(os.path.join(root, fn))
    return matches

ckpt_matches = find_unique_input_file(
    exact_names={"eff_full_best (6).pt", "eff_full_best.pt"}
)
if not ckpt_matches:
    ckpt_matches = find_unique_input_file(contains=["eff_full_best"], suffix=".pt")

tok_matches = find_unique_input_file(exact_names={"tokenizer.json"})
if not tok_matches:
    tok_matches = find_unique_input_file(contains=["tokenizer"], suffix=".json")

print("Checkpoint candidates:")
for p in ckpt_matches:
    print(" ", p)

print("\nTokenizer candidates:")
for p in tok_matches:
    print(" ", p)

assert ckpt_matches, "Could not find eff_full_best checkpoint under /kaggle/input."
assert tok_matches, "Could not find tokenizer.json under /kaggle/input."

# Prefer files from a path containing phase1/model4 if multiple are attached.
def choose_preferred(paths, keywords):
    scored = []
    for p in paths:
        low = p.lower()
        score = sum(k.lower() in low for k in keywords)
        scored.append((score, p))
    return sorted(scored, reverse=True)[0][1]

PHASE1_CKPT = choose_preferred(ckpt_matches, ["phase1", "model4", "decoder", "efficient"])
TOKENIZER_FILE = choose_preferred(tok_matches, ["phase1", "model4", "decoder", "efficient"])

print("\nUsing checkpoint:", PHASE1_CKPT)
print("Using tokenizer :", TOKENIZER_FILE)

Checkpoint candidates:
  /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt

Tokenizer candidates:
  /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json

Using checkpoint: /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
Using tokenizer : /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json


In [3]:
# ============================================================
# CELL 3 — Original custom byte-level LegalBPE
# ============================================================

BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            tok.symbol_bytes[s] = bytes([b])
        for l, r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

tokenizer = LegalBPE.load_json(TOKENIZER_FILE)
print("Tokenizer vocab:", len(tokenizer))
assert len(tokenizer) == VOCAB_SIZE, (len(tokenizer), VOCAB_SIZE)

probe = "IN THE SUPREME COURT OF INDIA\nSection 14: legal test."
assert tokenizer.decode(tokenizer.encode(probe)) == probe
print("✅ tokenizer round-trip passed")

Tokenizer vocab: 24576
✅ tokenizer round-trip passed


In [4]:
# ============================================================
# CELL 4 — Exact Phase-1 Model 4 architecture
# ============================================================

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return x32.to(dtype) * self.weight


class RotaryTable(nn.Module):
    def __init__(self, head_dim, max_positions, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", freqs.cos(), persistent=False)
        self.register_buffer("sin_cached", freqs.sin(), persistent=False)
        self.max_positions = max_positions

    def forward(self, position_ids):
        return (
            self.cos_cached[position_ids].unsqueeze(1),
            self.sin_cached[position_ids].unsqueeze(1),
        )


def apply_rope(x, cos, sin):
    B, H, T, D = x.shape
    cos = cos.to(x.dtype)
    sin = sin.to(x.dtype)
    xp = x.view(B, H, T, D // 2, 2)
    x0, x1 = xp[..., 0], xp[..., 1]
    return torch.stack(
        (x0 * cos - x1 * sin, x0 * sin + x1 * cos),
        dim=-1
    ).flatten(-2)


class GQACausalSelfAttention(nn.Module):
    GQA_DECODE_PATH = True

    def __init__(self, d_model, n_heads, n_kv_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_repeat = n_heads // n_kv_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape
        H, KV, D = self.n_heads, self.n_kv_heads, self.head_dim

        q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)
        k = self.k_proj(x).view(B, T, KV, D).transpose(1, 2)
        v = self.v_proj(x).view(B, T, KV, D).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        new_kv = (k, v) if use_cache else None
        S = k.size(2)
        drop = self.dropout if self.training else 0.0

        if T == 1 and past_len > 0 and self.GQA_DECODE_PATH:
            qg = q.reshape(B, KV, self.kv_repeat, D)
            mask = None if key_valid is None else key_valid[:, None, None, :]
            y = F.scaled_dot_product_attention(
                qg, k, v, attn_mask=mask, dropout_p=drop, is_causal=False
            )
            y = y.reshape(B, H, 1, D).transpose(1, 2).contiguous().view(B, 1, C)
            return self.o_proj(y), new_kv

        if self.kv_repeat != 1:
            k_attn = k.repeat_interleave(self.kv_repeat, dim=1)
            v_attn = v.repeat_interleave(self.kv_repeat, dim=1)
        else:
            k_attn, v_attn = k, v

        attn_mask, is_causal = None, False
        if T == 1:
            if key_valid is not None:
                attn_mask = key_valid[:, None, None, :]
        elif key_valid is None and past_len == 0:
            is_causal = True
        else:
            qi = torch.arange(T, device=x.device)[:, None] + past_len
            kj = torch.arange(S, device=x.device)[None, :]
            attn_mask = (kj <= qi)[None, None, :, :]
            if key_valid is not None:
                attn_mask = attn_mask & key_valid[:, None, None, :]
                if past_len == 0:
                    attn_mask = attn_mask | (kj == qi)[None, None, :, :]

        y = F.scaled_dot_product_attention(
            q, k_attn, v_attn,
            attn_mask=attn_mask,
            dropout_p=drop,
            is_causal=is_causal
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y), new_kv


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.up_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.dropout(self.down_proj(x))


class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps):
        super().__init__()
        self.norm1 = RMSNorm(d_model, rms_eps)
        self.attn = GQACausalSelfAttention(d_model, n_heads, n_kv_heads, dropout)
        self.norm2 = RMSNorm(d_model, rms_eps)
        self.ff = SwiGLU(d_model, ffn_dim, dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.norm1(x), cos, sin,
            past_kv=past_kv,
            use_cache=use_cache,
            key_valid=key_valid
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.norm2(x))
        return x, new_kv


class EfficientDecoderLM(nn.Module):
    def __init__(
        self, vocab_size, d_model, n_layers, n_heads, n_kv_heads,
        ffn_dim, dropout, max_positions, rope_base, rms_eps
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryTable(d_model // n_heads, max_positions, rope_base)

        self.blocks = nn.ModuleList([
            EfficientDecoderBlock(
                d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps
            )
            for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model, rms_eps)

        self.apply(self._init_weights)
        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.o_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff.down_proj.weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(
        self, input_ids, past_key_values=None, use_cache=False,
        position_ids=None, key_valid=None
    ):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len + T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=input_ids.device
            )[None, :].expand(B, T)

        cos, sin = self.rope(position_ids)

        x = self.drop(self.token_emb(input_ids))
        new_cache = [] if use_cache else None

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(
                x, cos, sin,
                past_kv=past,
                use_cache=use_cache,
                key_valid=key_valid
            )
            if use_cache:
                new_cache.append(kv)

        return self.final_norm(x), new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)


def build_backbone():
    return EfficientDecoderLM(
        vocab_size=len(tokenizer),
        d_model=D_MODEL,
        n_layers=N_LAYERS,
        n_heads=N_HEADS,
        n_kv_heads=N_KV_HEADS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        max_positions=MAX_POSITIONS,
        rope_base=ROPE_BASE,
        rms_eps=RMS_EPS,
    )

backbone = build_backbone()
print("Fresh backbone params:", f"{sum(p.numel() for p in backbone.parameters())/1e6:.3f}M")

Fresh backbone params: 99.309M


In [6]:
# ============================================================
# CELL 5 — Load Phase-1 Model 4 checkpoint exactly
# ============================================================

phase1_obj = torch.load(PHASE1_CKPT, map_location="cpu", weights_only=False)
print("Checkpoint keys:", list(phase1_obj.keys()))

state = phase1_obj["model"] if isinstance(phase1_obj, dict) and "model" in phase1_obj else phase1_obj

missing, unexpected = backbone.load_state_dict(state, strict=True)
assert not missing and not unexpected

print("✅ Phase-1 Model 4 weights loaded with strict=True")
print("global_step:", phase1_obj.get("global_step") if isinstance(phase1_obj, dict) else None)
print("val_loss:", phase1_obj.get("val_loss") if isinstance(phase1_obj, dict) else None)
del state
gc.collect()

Checkpoint keys: ['model', 'global_step', 'val_loss', 'config']
✅ Phase-1 Model 4 weights loaded with strict=True
global_step: 16266
val_loss: 1.3664915735571412


10

In [7]:
# ============================================================
# CELL 6 — Custom LoRA from scratch (attention + MLP projections)
# ============================================================

class LoRALinear(nn.Module):
    def __init__(self, base_linear, r=8, alpha=16, dropout=0.05):
        super().__init__()
        assert isinstance(base_linear, nn.Linear)
        assert r > 0

        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False

        self.r = int(r)
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.r
        self.dropout = nn.Dropout(dropout)

        self.lora_A = nn.Parameter(torch.empty(self.r, base_linear.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base_linear.out_features, self.r))

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base_out = self.base(x)
        lora_out = F.linear(F.linear(self.dropout(x), self.lora_A), self.lora_B)
        return base_out + self.scaling * lora_out


def freeze_all(module):
    for p in module.parameters():
        p.requires_grad = False


def inject_lora(model, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT, targets=LORA_TARGETS):
    freeze_all(model)
    replaced = []
    for i, block in enumerate(model.blocks):
        groups = (
            ("attn", block.attn, ["q_proj", "k_proj", "v_proj", "o_proj"]),
            ("ff", block.ff, ["gate_proj", "up_proj", "down_proj"]),
        )
        for gname, mod, names in groups:
            for nme in names:
                if nme in targets:
                    setattr(mod, nme, LoRALinear(getattr(mod, nme), r=r, alpha=alpha, dropout=dropout))
                    replaced.append(f"blocks.{i}.{gname}.{nme}")
    return replaced

inject_qv_lora = inject_lora   # compatibility alias used by the refit cell


replaced = inject_lora(backbone)
print("Injected LoRA modules:", len(replaced))
print("First 6:", replaced[:6])

lora_trainable = sum(
    p.numel() for p in backbone.parameters() if p.requires_grad
)
total_backbone = sum(p.numel() for p in backbone.parameters())

print("Backbone total params    :", f"{total_backbone/1e6:.3f}M")
print("LoRA trainable params    :", f"{lora_trainable/1e6:.3f}M")
print("LoRA trainable percentage:", f"{100*lora_trainable/total_backbone:.4f}%")

Injected LoRA modules: 119
First 6: ['blocks.0.attn.q_proj', 'blocks.0.attn.k_proj', 'blocks.0.attn.v_proj', 'blocks.0.attn.o_proj', 'blocks.0.ff.gate_proj', 'blocks.0.ff.up_proj']
Backbone total params    : 102.617M
LoRA trainable params    : 3.308M
LoRA trainable percentage: 3.2232%


In [8]:
# ============================================================
# CELL 7 — Download Phase-3 raw JSON files (bypass datasets CastError)
# ============================================================

api = HfApi()
repo_files = api.list_repo_files(
    repo_id=HF_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
)

def exact_repo_file(filename):
    matches = [
        f for f in repo_files
        if os.path.basename(f).lower() == filename.lower()
    ]
    assert len(matches) == 1, (filename, matches)
    return matches[0]

FILES = {
    "CL_train": exact_repo_file("CL_train.json"),
    "CL_dev": exact_repo_file("CL_dev.json"),
    "IT_train": exact_repo_file("IT_train.json"),
    "IT_dev": exact_repo_file("IT_dev.json"),
    "CL_test": exact_repo_file("CL_test.json"),
    "IT_test": exact_repo_file("IT_test.json"),
    "label_vocab": exact_repo_file("label_vocab.json"),
}

print(json.dumps(FILES, indent=2))

def load_repo_json(repo_path):
    p = hf_hub_download(
        repo_id=HF_REPO,
        filename=repo_path,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

cl_train_raw = load_repo_json(FILES["CL_train"])
cl_dev_raw   = load_repo_json(FILES["CL_dev"])
it_train_raw = load_repo_json(FILES["IT_train"])
it_dev_raw   = load_repo_json(FILES["IT_dev"])
cl_test_raw  = load_repo_json(FILES["CL_test"])
it_test_raw  = load_repo_json(FILES["IT_test"])
label_vocab_raw = load_repo_json(FILES["label_vocab"])

assert isinstance(cl_train_raw, list) and isinstance(it_train_raw, list)
assert isinstance(cl_dev_raw, list) and isinstance(it_dev_raw, list)
assert isinstance(cl_test_raw, dict) and isinstance(it_test_raw, dict)

print("CL train/dev:", len(cl_train_raw), len(cl_dev_raw))
print("IT train/dev:", len(it_train_raw), len(it_dev_raw))
print("CL/IT test docs:", len(cl_test_raw), len(it_test_raw))

{
  "CL_train": "Phase_3_RR/Train/CL_train.json",
  "CL_dev": "Phase_3_RR/Train/CL_dev.json",
  "IT_train": "Phase_3_RR/Train/IT_train.json",
  "IT_dev": "Phase_3_RR/Train/IT_dev.json",
  "CL_test": "Phase_3_RR/Test/CL_test.json",
  "IT_test": "Phase_3_RR/Test/IT_test.json",
  "label_vocab": "Phase_3_RR/Train/label_vocab.json"
}


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL train/dev: 40 5
IT train/dev: 40 5
CL/IT test docs: 5 5


In [9]:
# ============================================================
# CELL 8 — Labels, examples, class weights, and data checks
# ============================================================

def parse_label_vocab(data):
    if isinstance(data, list):
        labels = list(data)
        label_to_id = {x:i for i,x in enumerate(labels)}
    elif isinstance(data, dict):
        vals = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in vals):
            label_to_id = {str(k): int(v) for k, v in data.items()}
            assert sorted(label_to_id.values()) == list(range(len(label_to_id)))
            labels = [None] * len(label_to_id)
            for lab, idx in label_to_id.items():
                labels[idx] = lab
        elif all(isinstance(v, str) for v in vals):
            tmp = {int(k): v for k, v in data.items()}
            labels = [tmp[i] for i in sorted(tmp)]
            label_to_id = {x:i for i,x in enumerate(labels)}
        else:
            raise ValueError("Unsupported label_vocab format")
    else:
        raise ValueError("Unsupported label_vocab format")

    return labels, label_to_id

LABELS, LABEL_TO_ID = parse_label_vocab(label_vocab_raw)
ID_TO_LABEL = {v:k for k,v in LABEL_TO_ID.items()}
NUM_LABELS = len(LABELS)

assert NUM_LABELS == 13
print("Labels:")
for i, x in enumerate(LABELS):
    print(i, x)

def normalize_id(doc_id):
    doc_id = str(doc_id)
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id

def validate_labeled_docs(docs, name):
    ids = set()
    n_sent = 0
    for d in docs:
        did = normalize_id(d["id"])
        assert did not in ids, f"{name}: duplicate ID {did}"
        ids.add(did)
        assert isinstance(d["text"], list)
        assert isinstance(d["labels"], list)
        assert len(d["text"]) == len(d["labels"]), did
        assert all(x in LABEL_TO_ID for x in d["labels"]), did
        n_sent += len(d["text"])
    print(f"{name}: docs={len(docs)} sentences={n_sent}")
    return ids

cltr_ids = validate_labeled_docs(cl_train_raw, "CL_train")
cldv_ids = validate_labeled_docs(cl_dev_raw, "CL_dev")
ittr_ids = validate_labeled_docs(it_train_raw, "IT_train")
itdv_ids = validate_labeled_docs(it_dev_raw, "IT_dev")

assert not (cltr_ids & cldv_ids)
assert not (ittr_ids & itdv_ids)
assert not (cldv_ids & itdv_ids)

def position_features(i, n):
    if not USE_POS:
        return [0.0, 0.0, 0.0, 0.0]
    return [i / max(1, n - 1), math.log1p(n) / 8.0, float(i == 0), float(i == n - 1)]

# Flatten into sentence-level records while preserving document context.
def make_labeled_records(docs, domain):
    out = []
    for d in docs:
        sents = d["text"]
        labs = d["labels"]
        did = normalize_id(d["id"])
        for i, (sent, lab) in enumerate(zip(sents, labs)):
            out.append({
                "doc_id": did,
                "sent_idx": i,
                "domain": domain,
                "prev": " ".join(sents[max(0, i-CONTEXT_SENTS):i]) if CONTEXT_SENTS > 0 else "",
                "target": sent,
                "next": " ".join(sents[i+1:i+1+CONTEXT_SENTS]) if CONTEXT_SENTS > 0 else "",
                "pos": position_features(i, len(sents)),
                "label": lab,
                "label_id": LABEL_TO_ID[lab],
            })
    return out

train_records = make_labeled_records(cl_train_raw, "CL") + make_labeled_records(it_train_raw, "IT")
dev_records   = make_labeled_records(cl_dev_raw, "CL") + make_labeled_records(it_dev_raw, "IT")

print("Train sentence records:", len(train_records))
print("Dev sentence records  :", len(dev_records))

train_counts = Counter(x["label"] for x in train_records)
print("\nTrain distribution:")
for lab in LABELS:
    print(f"{lab:30s} {train_counts[lab]:6d}")

# sqrt inverse-frequency; normalize mean to 1; then cap extremes.
N = len(train_records)
C = NUM_LABELS
raw_w = np.array(
    [math.sqrt(N / (C * max(train_counts[lab], 1))) for lab in LABELS],
    dtype=np.float32
)
raw_w = raw_w / raw_w.mean()
class_w = np.clip(raw_w, CLASS_WEIGHT_MIN, CLASS_WEIGHT_MAX)
class_w = torch.tensor(class_w, dtype=torch.float32, device=DEVICE)

print("\nClass weights:")
for lab, w in zip(LABELS, class_w.detach().cpu().tolist()):
    print(f"{lab:30s} {w:.4f}")

Labels:
0 Fact
1 Issue
2 ArgumentPetitioner
3 ArgumentRespondent
4 Statute
5 Dissent
6 PrecedentReliedUpon
7 PrecedentNotReliedUpon
8 PrecedentOverruled
9 RulingByLowerCourt
10 RatioOfTheDecision
11 RulingByPresentCourt
12 None
CL_train: docs=40 sentences=10593
CL_dev: docs=5 sentences=1396
IT_train: docs=40 sentences=6252
IT_dev: docs=5 sentences=746
Train sentence records: 16845
Dev sentence records  : 2142

Train distribution:
Fact                             4098
Issue                             107
ArgumentPetitioner               1613
ArgumentRespondent               1348
Statute                          1099
Dissent                           284
PrecedentReliedUpon              1945
PrecedentNotReliedUpon            569
PrecedentOverruled                  3
RulingByLowerCourt                874
RatioOfTheDecision               4301
RulingByPresentCourt              307
None                              297

Class weights:
Fact                           0.3500
Issue             

In [10]:
# ============================================================
# CELL 9 — Context construction and tokenization
# ============================================================

_ENC_CACHE = {}
def enc(text):
    hit = _ENC_CACHE.get(text)
    if hit is None:
        hit = tokenizer.encode(text)
        _ENC_CACHE[text] = hit
    return hit

def crop_head_tail(ids, budget):
    if len(ids) <= budget:
        return ids
    if budget <= 0:
        return []
    left = budget // 2
    right = budget - left
    return ids[:left] + ids[-right:]

def build_context_ids(domain, prev_text, target_text, next_text, max_len=MAX_LEN):
    # Ordinary text markers only: no tokenizer-vocabulary modification.
    prefix = enc(f"Domain: {domain}\n")
    pmark  = enc("Previous sentence:\n")
    tmark  = enc("\nTarget sentence:\n")
    nmark  = enc("\nNext sentence:\n")
    rmark  = enc("\nRhetorical role:")

    prev_ids = enc(prev_text) if prev_text else []
    targ_ids = enc(target_text)
    next_ids = enc(next_text) if next_text else []

    # Initial caps protect the target sentence and preserve both neighboring sides.
    prev_ids = prev_ids[-PREV_BUDGET:]
    targ_ids = crop_head_tail(targ_ids, TARGET_BUDGET)
    next_ids = next_ids[:NEXT_BUDGET]

    fixed = prefix + pmark + tmark + nmark + rmark
    remaining = max_len - len(fixed)

    if remaining < 8:
        raise RuntimeError("Marker text unexpectedly consumes MAX_LEN.")

    # Fit segments to remaining space, with target priority.
    t_keep = min(len(targ_ids), max(8, min(TARGET_BUDGET, remaining)))
    remaining2 = remaining - t_keep

    p_keep = min(len(prev_ids), remaining2 // 2)
    n_keep = min(len(next_ids), remaining2 - p_keep)

    # Give any spare capacity back to previous/next, then target if available.
    spare = remaining2 - p_keep - n_keep
    if spare > 0:
        add_p = min(spare, len(prev_ids) - p_keep)
        p_keep += add_p
        spare -= add_p
    if spare > 0:
        add_n = min(spare, len(next_ids) - n_keep)
        n_keep += add_n
        spare -= add_n

    prev_ids = prev_ids[-p_keep:] if p_keep else []
    targ_ids = crop_head_tail(targ_ids, t_keep)
    next_ids = next_ids[:n_keep] if n_keep else []

    ids = (
        prefix
        + pmark + prev_ids
        + tmark + targ_ids
        + nmark + next_ids
        + rmark
    )

    if len(ids) > max_len:
        # Absolute safety fallback; preserve the terminal role marker.
        ids = ids[-max_len:]

    assert 1 <= len(ids) <= max_len
    return ids

# Token-length audit
lengths = [
    len(build_context_ids(
        r["domain"], r["prev"], r["target"], r["next"]
    ))
    for r in (train_records[:1000] + dev_records[:500])
]

print("Context-token sample:")
print(" mean:", round(float(np.mean(lengths)), 2))
print(" p95 :", round(float(np.percentile(lengths, 95)), 2))
print(" max :", max(lengths))

Context-token sample:
 mean: 317.94
 p95 : 404.0
 max : 594


In [11]:
# ============================================================
# CELL 10 — Dataset, collator, RR model
# ============================================================

class RRDataset(Dataset):
    def __init__(self, records, labeled=True):
        self.records = records
        self.labeled = labeled

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        ids = build_context_ids(
            r["domain"], r["prev"], r["target"], r["next"]
        )
        item = {
            "input_ids": ids,
            "doc_id": r["doc_id"],
            "sent_idx": r["sent_idx"],
            "domain": r["domain"],
            "pos": r["pos"],
        }
        if self.labeled:
            item["label_id"] = r["label_id"]
        return item


def collate_rr(batch):
    max_t = max(len(x["input_ids"]) for x in batch)
    B = len(batch)

    input_ids = torch.full(
        (B, max_t),
        tokenizer.pad_token_id,
        dtype=torch.long
    )
    key_valid = torch.zeros((B, max_t), dtype=torch.bool)
    lengths = torch.zeros(B, dtype=torch.long)
    pos = torch.tensor([x["pos"] for x in batch], dtype=torch.float32)

    for i, x in enumerate(batch):
        ids = torch.tensor(x["input_ids"], dtype=torch.long)
        L = len(ids)
        input_ids[i, :L] = ids
        key_valid[i, :L] = True
        lengths[i] = L

    out = {
        "input_ids": input_ids,
        "key_valid": key_valid,
        "lengths": lengths,
        "pos": pos,
        "doc_id": [x["doc_id"] for x in batch],
        "sent_idx": [x["sent_idx"] for x in batch],
        "domain": [x["domain"] for x in batch],
    }

    if "label_id" in batch[0]:
        out["labels"] = torch.tensor(
            [x["label_id"] for x in batch],
            dtype=torch.long
        )

    return out


HEAD_PREFIXES = ("head_norm.", "pos_mlp.", "fc1.", "classifier.")

class RRClassifier(nn.Module):
    def __init__(self, backbone, num_labels=13, dropout=0.15):
        super().__init__()
        self.backbone = backbone
        self.head_norm = RMSNorm(2 * D_MODEL, RMS_EPS)
        self.pos_mlp = nn.Sequential(nn.Linear(4, 64), nn.GELU())
        self.head_drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(2 * D_MODEL + 64, 512)
        self.classifier = nn.Linear(512, num_labels)

    def forward(self, input_ids, key_valid, lengths, pos):
        h, _ = self.backbone.forward_hidden(
            input_ids, key_valid=key_valid, use_cache=False
        )
        h = h.float()
        row = torch.arange(h.size(0), device=h.device)
        last_h = h[row, lengths - 1]                      # sees whole context (causal) after the role marker
        m = key_valid.unsqueeze(-1).to(h.dtype)
        mean_h = (h * m).sum(1) / m.sum(1).clamp(min=1)
        x = self.head_norm(torch.cat([last_h, mean_h], dim=-1))
        z = torch.cat([x, self.pos_mlp(pos.float())], dim=-1)
        z = F.gelu(self.fc1(self.head_drop(z)))
        return self.classifier(self.head_drop(z))


model = RRClassifier(backbone, NUM_LABELS, HEAD_DROPOUT).to(DEVICE)

# Classifier is trainable; backbone base is frozen except LoRA A/B.
for n_, p in model.named_parameters():
    if n_.startswith(HEAD_PREFIXES):
        p.requires_grad = True

trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
n_train = sum(x[1] for x in trainable)
n_total = sum(p.numel() for p in model.parameters())

print("Total params    :", f"{n_total/1e6:.3f}M")
print("Trainable params:", f"{n_train/1e6:.3f}M")
print("Trainable %     :", f"{100*n_train/n_total:.4f}%")
print("Trainable tensors:", len(trainable))
print("First trainable names:")
for x in trainable[:12]:
    print(" ", x)

Total params    : 103.314M
Trainable params: 4.004M
Trainable %     : 3.8760%
Trainable tensors: 245
First trainable names:
  ('backbone.blocks.0.attn.q_proj.lora_A', 10240)
  ('backbone.blocks.0.attn.q_proj.lora_B', 10240)
  ('backbone.blocks.0.attn.k_proj.lora_A', 10240)
  ('backbone.blocks.0.attn.k_proj.lora_B', 2048)
  ('backbone.blocks.0.attn.v_proj.lora_A', 10240)
  ('backbone.blocks.0.attn.v_proj.lora_B', 2048)
  ('backbone.blocks.0.attn.o_proj.lora_A', 10240)
  ('backbone.blocks.0.attn.o_proj.lora_B', 10240)
  ('backbone.blocks.0.ff.gate_proj.lora_A', 10240)
  ('backbone.blocks.0.ff.gate_proj.lora_B', 32768)
  ('backbone.blocks.0.ff.up_proj.lora_A', 10240)
  ('backbone.blocks.0.ff.up_proj.lora_B', 32768)


In [12]:
# ============================================================
# CELL 11 — Dataloaders, optimizer, scheduler
# ============================================================

train_loader = DataLoader(
    RRDataset(train_records, labeled=True),
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_rr,
    num_workers=2,
    pin_memory=True,
)

dev_loader = DataLoader(
    RRDataset(dev_records, labeled=True),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_rr,
    num_workers=2,
    pin_memory=True,
)

lora_params = []
head_params = []

for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if "lora_A" in name or "lora_B" in name:
        lora_params.append(p)
    else:
        head_params.append(p)

optimizer = torch.optim.AdamW(
    [
        {"params": lora_params, "lr": LORA_LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_updates = NUM_EPOCHS * updates_per_epoch
warmup_updates = max(1, int(total_updates * WARMUP_RATIO))

def lr_mult(step):
    if step < warmup_updates:
        return max(1e-8, step / warmup_updates)
    progress = (step - warmup_updates) / max(1, total_updates - warmup_updates)
    return 0.5 * (1.0 + math.cos(math.pi * min(max(progress, 0.0), 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_mult)

criterion = nn.CrossEntropyLoss(weight=class_w)

amp_enabled = bool(USE_AMP and DEVICE.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

print("Train batches/epoch:", len(train_loader))
print("Optimizer updates/epoch:", updates_per_epoch)
print("Total updates:", total_updates)
print("Warmup updates:", warmup_updates)
print("AMP enabled:", amp_enabled)

Train batches/epoch: 4212
Optimizer updates/epoch: 1053
Total updates: 6318
Warmup updates: 379
AMP enabled: True


In [14]:
# ============================================================
# CELL 12 — Metrics and evaluation
# ============================================================

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()

    all_true = []
    all_pred = []
    all_meta = []

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        key_valid = batch["key_valid"].to(DEVICE, non_blocking=True)
        lengths = batch["lengths"].to(DEVICE, non_blocking=True)
        pos = batch["pos"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            logits = model(input_ids, key_valid, lengths, pos)

        pred = logits.float().argmax(-1).cpu().tolist()
        all_pred.extend(pred)

        if "labels" in batch:
            all_true.extend(batch["labels"].tolist())

        all_meta.extend(zip(
            batch["doc_id"],
            batch["sent_idx"],
            batch["domain"]
        ))

    return all_true, all_pred, all_meta


def score_ids(y_true, y_pred, title="Combined"):
    label_ids = list(range(NUM_LABELS))

    macro_f1 = f1_score(
        y_true, y_pred,
        labels=label_ids,
        average="macro",
        zero_division=0
    )
    macro_p = precision_score(
        y_true, y_pred,
        labels=label_ids,
        average="macro",
        zero_division=0
    )
    macro_r = recall_score(
        y_true, y_pred,
        labels=label_ids,
        average="macro",
        zero_division=0
    )

    print(f"{title}: F1={macro_f1:.6f} P={macro_p:.6f} R={macro_r:.6f}")
    return macro_f1, macro_p, macro_r



def dev_label_ceiling_diagnostic(y_true):
    present = sorted(set(y_true))
    absent = [i for i in range(NUM_LABELS) if i not in present]
    ceiling = len(present) / NUM_LABELS
    print(
        f"Dev contains {len(present)}/{NUM_LABELS} labels; "
        f"formal Macro-F1 ceiling from label absence = {ceiling:.6f}"
    )
    if absent:
        print("Absent dev labels:", [LABELS[i] for i in absent])
    return ceiling


def detailed_dev_eval(model):
    y_true, y_pred, meta = predict_loader(model, dev_loader)

    combined = score_ids(y_true, y_pred, "Combined CL+IT")
    dev_label_ceiling_diagnostic(y_true)

    # ---- diagnostics: collapse check + F1 over labels actually present in dev
    pred_c, gold_c = Counter(y_pred), Counter(y_true)
    present = sorted(set(y_true))
    f1_present = f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)
    print(f"   Macro-F1 over {len(present)} labels present in dev: {f1_present:.6f}")
    print("   gold counts:", {LABELS[k]: gold_c[k] for k in present})
    print("   pred counts:", {LABELS[k]: pred_c[k] for k in sorted(pred_c)})
    if max(pred_c.values()) / max(1, len(y_pred)) > 0.60:
        print("   ⚠️ >60% of predictions are one class — model may be collapsing.")

    for dom in ["CL", "IT"]:
        idx = [i for i, x in enumerate(meta) if x[2] == dom]
        yt = [y_true[i] for i in idx]
        yp = [y_pred[i] for i in idx]
        score_ids(yt, yp, dom)

    print("\nPer-label report (all 13 labels explicitly included):")
    print(classification_report(
        y_true, y_pred,
        labels=list(range(NUM_LABELS)),
        target_names=LABELS,
        digits=4,
        zero_division=0
    ))

    return combined, y_true, y_pred, meta

@torch.no_grad()
def predict_logits(model, loader):
    """Returns (logits[N,13] float32, meta list, y_true list) in loader order."""
    model.eval()
    outs, meta, y_true = [], [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        key_valid = batch["key_valid"].to(DEVICE, non_blocking=True)
        lengths = batch["lengths"].to(DEVICE, non_blocking=True)
        pos = batch["pos"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            lg = model(input_ids, key_valid, lengths, pos)
        outs.append(lg.float().cpu().numpy())
        meta.extend(zip(batch["doc_id"], batch["sent_idx"], batch["domain"]))
        if "labels" in batch:
            y_true.extend(batch["labels"].tolist())
    return np.concatenate(outs, 0), meta, y_true


In [15]:
# ============================================================
# CELL 13 — Train LoRA model; select checkpoint by combined Macro-F1
# ============================================================

best_f1 = -1.0
best_epoch = None
bad_epochs = 0
global_update = 0
history = []

optimizer.zero_grad(set_to_none=True)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    seen_batches = 0
    t0 = time.time()

    for step, batch in enumerate(train_loader, start=1):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        key_valid = batch["key_valid"].to(DEVICE, non_blocking=True)
        lengths = batch["lengths"].to(DEVICE, non_blocking=True)
        pos = batch["pos"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            logits = model(input_ids, key_valid, lengths, pos)
            loss = criterion(logits, labels)
            scaled_loss = loss / GRAD_ACCUM_STEPS

        if not torch.isfinite(loss.detach()):
            raise FloatingPointError(f"Non-finite loss at epoch={epoch}, batch={step}")
        scaler.scale(scaled_loss).backward()

        running_loss += float(loss.detach().cpu())
        seen_batches += 1

        do_update = (
            step % GRAD_ACCUM_STEPS == 0
            or step == len(train_loader)
        )

        if do_update:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                GRAD_CLIP
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            scheduler.step()
            global_update += 1

        if step % 250 == 0 or step == len(train_loader):
            print(
                f"epoch={epoch} batch={step}/{len(train_loader)} "
                f"loss={running_loss/max(seen_batches,1):.4f} "
                f"lr_lora={optimizer.param_groups[0]['lr']:.2e} "
                f"lr_head={optimizer.param_groups[1]['lr']:.2e}",
                flush=True
            )

    (metrics, y_true, y_pred, meta) = detailed_dev_eval(model)
    macro_f1, macro_p, macro_r = metrics

    epoch_row = {
        "epoch": epoch,
        "train_loss": running_loss / max(seen_batches, 1),
        "macro_f1": macro_f1,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "elapsed_sec": time.time() - t0,
    }
    history.append(epoch_row)

    print("Epoch summary:", epoch_row)

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        best_epoch = epoch

        # Save only task-adaptation state plus metadata.
        task_state = {
            k: v.detach().cpu()
            for k, v in model.state_dict().items()
            if (
                "lora_A" in k
                or "lora_B" in k
                or k.startswith(HEAD_PREFIXES)
            )
        }

        torch.save({
            "task_state": task_state,
            "best_epoch": best_epoch,
            "best_macro_f1": best_f1,
            "history": history,
            "labels": LABELS,
            "label_to_id": LABEL_TO_ID,
            "config": {
                "max_len": MAX_LEN,
                "lora_r": LORA_R,
                "lora_alpha": LORA_ALPHA,
                "lora_dropout": LORA_DROPOUT,
                "targets": LORA_TARGETS,
                "context": f"+-{CONTEXT_SENTS} sentences + domain + position(use_pos={USE_POS})",
                "class_weight_min": CLASS_WEIGHT_MIN,
                "class_weight_max": CLASS_WEIGHT_MAX,
            }
        }, BEST_CKPT)

        print("✅ NEW BEST:", BEST_CKPT)

    print(f"Best so far: epoch={best_epoch} Macro-F1={best_f1:.6f}\n")

    bad_epochs = 0 if best_epoch == epoch else bad_epochs + 1
    if bad_epochs >= EARLY_STOP_PATIENCE:
        print(f"⏹️ Early stopping: no dev improvement for {EARLY_STOP_PATIENCE} epochs.")
        break

print("Training complete.")
print("BEST_EPOCH =", best_epoch)
print("BEST_MACRO_F1 =", best_f1)

epoch=1 batch=250/4212 loss=2.2464 lr_lora=3.27e-05 lr_head=1.64e-04
epoch=1 batch=500/4212 loss=2.0643 lr_lora=6.60e-05 lr_head=3.30e-04
epoch=1 batch=750/4212 loss=1.8816 lr_lora=9.87e-05 lr_head=4.93e-04
epoch=1 batch=1000/4212 loss=1.7772 lr_lora=1.32e-04 lr_head=6.60e-04
epoch=1 batch=1250/4212 loss=1.6974 lr_lora=1.65e-04 lr_head=8.23e-04
epoch=1 batch=1500/4212 loss=1.6221 lr_lora=1.98e-04 lr_head=9.89e-04
epoch=1 batch=1750/4212 loss=1.5561 lr_lora=2.00e-04 lr_head=1.00e-03
epoch=1 batch=2000/4212 loss=1.5038 lr_lora=2.00e-04 lr_head=9.99e-04
epoch=1 batch=2250/4212 loss=1.4515 lr_lora=2.00e-04 lr_head=9.98e-04
epoch=1 batch=2500/4212 loss=1.4107 lr_lora=1.99e-04 lr_head=9.96e-04
epoch=1 batch=2750/4212 loss=1.3747 lr_lora=1.99e-04 lr_head=9.93e-04
epoch=1 batch=3000/4212 loss=1.3370 lr_lora=1.98e-04 lr_head=9.90e-04
epoch=1 batch=3250/4212 loss=1.3030 lr_lora=1.97e-04 lr_head=9.87e-04
epoch=1 batch=3500/4212 loss=1.2744 lr_lora=1.97e-04 lr_head=9.83e-04
epoch=1 batch=3750/4212

In [25]:
# ============================================================
# CELL 14 — Reload BEST checkpoint and reconstruct dev prediction.json
# ============================================================

best_obj = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)

missing, unexpected = model.load_state_dict(
    best_obj["task_state"],
    strict=False
)

# Expected missing keys are frozen Phase-1 base weights.
bad_unexpected = [x for x in unexpected]
assert not bad_unexpected, bad_unexpected

model.to(DEVICE)
model.eval()

(metrics, y_true, y_pred, meta) = detailed_dev_eval(model)
reloaded_f1 = metrics[0]

print("Stored best F1 :", best_obj["best_macro_f1"])
print("Reloaded F1    :", reloaded_f1)
print("Delta          :", abs(best_obj["best_macro_f1"] - reloaded_f1))

assert abs(best_obj["best_macro_f1"] - reloaded_f1) < 5e-3, \
    "Reloaded checkpoint does not reproduce stored dev F1."

# Convert flat predictions back to document -> ordered label list.
def records_to_prediction(meta, pred_ids):
    temp = {}
    for (doc_id, sent_idx, domain), pred_id in zip(meta, pred_ids):
        temp.setdefault(doc_id, []).append((sent_idx, ID_TO_LABEL[int(pred_id)]))

    out = {}
    for doc_id, pairs in temp.items():
        pairs.sort(key=lambda x: x[0])
        out[doc_id] = [lab for _, lab in pairs]
    return out

dev_prediction = records_to_prediction(meta, y_pred)

with open(DEV_PRED, "w", encoding="utf-8") as f:
    json.dump(dev_prediction, f, ensure_ascii=False, indent=2)

def truth_map(docs):
    return {
        normalize_id(d["id"]): list(d["labels"])
        for d in docs
    }

with open(CL_DEV_TRUTH, "w", encoding="utf-8") as f:
    json.dump(truth_map(cl_dev_raw), f, ensure_ascii=False, indent=2)

with open(IT_DEV_TRUTH, "w", encoding="utf-8") as f:
    json.dump(truth_map(it_dev_raw), f, ensure_ascii=False, indent=2)

# Save exact public label vocabulary; evaluator accepts label->id mapping.
with open(LABELS_JSON, "w", encoding="utf-8") as f:
    json.dump(label_vocab_raw, f, ensure_ascii=False, indent=2)

print("Saved:")
print(CL_DEV_TRUTH)
print(IT_DEV_TRUTH)
print(DEV_PRED)
print(LABELS_JSON)

Combined CL+IT: F1=0.430887 P=0.443901 R=0.423628
Dev contains 11/13 labels; formal Macro-F1 ceiling from label absence = 0.846154
Absent dev labels: ['Dissent', 'PrecedentOverruled']
   Macro-F1 over 11 labels present in dev: 0.509230
   gold counts: {'Fact': 444, 'Issue': 15, 'ArgumentPetitioner': 400, 'ArgumentRespondent': 272, 'Statute': 120, 'PrecedentReliedUpon': 254, 'PrecedentNotReliedUpon': 50, 'RulingByLowerCourt': 102, 'RatioOfTheDecision': 403, 'RulingByPresentCourt': 49, 'None': 33}
   pred counts: {'Fact': 509, 'Issue': 12, 'ArgumentPetitioner': 345, 'ArgumentRespondent': 208, 'Statute': 122, 'Dissent': 1, 'PrecedentReliedUpon': 244, 'PrecedentNotReliedUpon': 57, 'RulingByLowerCourt': 81, 'RatioOfTheDecision': 489, 'RulingByPresentCourt': 48, 'None': 26}
CL: F1=0.384458 P=0.375594 R=0.421152
IT: F1=0.429032 P=0.476391 R=0.411711

Per-label report (all 13 labels explicitly included):
                        precision    recall  f1-score   support

                  Fact   

In [26]:
# ============================================================
# CELL 15 — EXACT local Phase-3 evaluator script supplied by course
# ============================================================

EVALUATOR_CODE = r'''#!/usr/bin/env python3
"""Local evaluator for Phase 3.

The competition scorer evaluates two gold/reference JSON files together and
one participant prediction.json. This local evaluator mirrors that setup
using the student's own held-out splits.
"""
import argparse
import json
import sys

from sklearn.metrics import f1_score, precision_score, recall_score

EXPECTED_LABEL_COUNT = 13


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def normalize_id(doc_id):
    if not isinstance(doc_id, str) or not doc_id.strip():
        raise ValueError("Document IDs must be non-empty strings.")
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id


def load_vocab(data):
    if isinstance(data, list):
        labels = data
    elif isinstance(data, dict) and data:
        values = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in values):
            labels = list(data.keys())
        elif all(isinstance(v, str) for v in values):
            labels = values
        else:
            raise ValueError("Unsupported labels.json format.")
    else:
        raise ValueError("labels.json must be a non-empty list or object.")

    if len(labels) != EXPECTED_LABEL_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_LABEL_COUNT} labels, found {len(labels)}."
        )
    if any(not isinstance(label, str) or not label.strip() for label in labels):
        raise ValueError("Labels must be unique non-empty strings.")
    if len(set(labels)) != len(labels):
        raise ValueError("Labels must be unique non-empty strings.")
    return labels


def normalize_reference(data, source_name):
    if not isinstance(data, dict):
        raise ValueError(f"{source_name}: reference data must be a JSON object.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"{source_name}: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def normalize_predictions(data):
    if not isinstance(data, dict):
        raise ValueError("prediction.json must be a JSON object mapping document IDs to label lists.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"prediction.json: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Evaluate Phase 3 sentence-label predictions on two local held-out "
            "reference splits, mirroring the competition's combined scoring setup."
        )
    )
    parser.add_argument("--truth-cl", required=True)
    parser.add_argument("--truth-it", required=True)
    parser.add_argument("--prediction", required=True)
    parser.add_argument("--labels", required=True)
    args = parser.parse_args()

    truth_cl = normalize_reference(load_json(args.truth_cl), "--truth-cl")
    truth_it = normalize_reference(load_json(args.truth_it), "--truth-it")
    predictions = normalize_predictions(load_json(args.prediction))
    labels = load_vocab(load_json(args.labels))
    allowed = set(labels)

    overlap = set(truth_cl) & set(truth_it)
    if overlap:
        sample = sorted(overlap)[:5]
        suffix = "..." if len(overlap) > 5 else ""
        raise ValueError(
            "The two reference splits contain overlapping document IDs: "
            + ", ".join(sample)
            + suffix
        )

    truth = {}
    truth.update(truth_cl)
    truth.update(truth_it)

    if set(truth) != set(predictions):
        missing = sorted(set(truth) - set(predictions))
        extra = sorted(set(predictions) - set(truth))
        parts = [f"missing={len(missing)}", f"extra={len(extra)}"]
        if missing:
            parts.append(f"example missing ID(s)={missing[:5]}")
        if extra:
            parts.append(f"example extra ID(s)={extra[:5]}")
        raise ValueError("ID mismatch: " + "; ".join(parts))

    y_true = []
    y_pred = []

    for doc_id in truth:
        gold = truth[doc_id]
        pred = predictions[doc_id]

        if not isinstance(gold, list) or not isinstance(pred, list):
            raise ValueError(f"{doc_id}: labels must be JSON lists.")
        if len(gold) != len(pred):
            raise ValueError(
                f"{doc_id}: sentence-count mismatch ({len(gold)} vs {len(pred)})."
            )
        if any(label not in allowed for label in gold + pred):
            raise ValueError(f"{doc_id}: unknown label encountered.")

        y_true.extend(gold)
        y_pred.extend(pred)

    score = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_precision = precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    print(f"CL documents: {len(truth_cl)}")
    print(f"IT documents: {len(truth_it)}")
    print(f"Total documents: {len(truth)}")
    print(f"Total sentences: {len(y_true)}")
    print(f"Macro-F1:        {score:.6f}")
    print(f"Macro-Precision: {macro_precision:.6f}")
    print(f"Macro-Recall:    {macro_recall:.6f}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"EVALUATION FAILED: {exc}", file=sys.stderr)
        sys.exit(1)
'''

with open(EVAL_SCRIPT, "w", encoding="utf-8") as f:
    f.write(EVALUATOR_CODE)

print("Saved exact evaluator to:", EVAL_SCRIPT)

Saved exact evaluator to: /kaggle/working/phase3_exp1_model4_lora/evaluate_phase3.py


In [27]:
# ============================================================
# CELL 16 — RUN THE OFFICIAL-STYLE LOCAL EVALUATOR
# Do not continue to final refit until this passes.
# ============================================================

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--truth-cl", str(CL_DEV_TRUTH),
    "--truth-it", str(IT_DEV_TRUTH),
    "--prediction", str(DEV_PRED),
    "--labels", str(LABELS_JSON),
]

print("Running:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("\nSTDOUT:\n", result.stdout)
if result.stderr:
    print("\nSTDERR:\n", result.stderr)

assert result.returncode == 0, "❌ Phase-3 evaluator failed."
print("✅ DEV prediction passed the exact Phase-3 evaluator.")

Running:
/usr/bin/python3 /kaggle/working/phase3_exp1_model4_lora/evaluate_phase3.py --truth-cl /kaggle/working/phase3_exp1_model4_lora/cl_test_truth.json --truth-it /kaggle/working/phase3_exp1_model4_lora/it_test_truth.json --prediction /kaggle/working/phase3_exp1_model4_lora/prediction_dev.json --labels /kaggle/working/phase3_exp1_model4_lora/labels.json

STDOUT:
 CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.430887
Macro-Precision: 0.443901
Macro-Recall:    0.423628

✅ DEV prediction passed the exact Phase-3 evaluator.


In [28]:
# ============================================================
# CELL 16b — Optional transition-aware decoding tuned on DEV
#
# Improvements:
#   * CL and IT transition/prior statistics estimated separately
#   * additive smoothing
#   * no hard "impossible label" bans
#   * only adopt decoding when DEV Macro-F1 improves by >= 0.005
# ============================================================

from sklearn.metrics import f1_score as _f1

def transition_matrix(docs_list, smooth=1.0):
    T = np.full((NUM_LABELS, NUM_LABELS), float(smooth), dtype=np.float64)
    for d in docs_list:
        ids = [LABEL_TO_ID[x] for x in d["labels"]]
        for a, b in zip(ids[:-1], ids[1:]):
            T[a, b] += 1.0
    T /= T.sum(1, keepdims=True)
    return np.log(T)

def class_log_prior(docs_list, smooth=1.0):
    c = np.full(NUM_LABELS, float(smooth), dtype=np.float64)
    for d in docs_list:
        for x in d["labels"]:
            c[LABEL_TO_ID[x]] += 1.0
    return np.log(c / c.sum())

def log_softmax_np(x):
    x = x - x.max(-1, keepdims=True)
    return x - np.log(np.exp(x).sum(-1, keepdims=True))

def viterbi_decode(logp, trans, beta):
    n, K = logp.shape
    score = logp[0].copy()
    back = np.zeros((n, K), dtype=np.int32)
    tr = beta * trans

    for t in range(1, n):
        cand = score[:, None] + tr
        back[t] = cand.argmax(0)
        score = cand[back[t], np.arange(K)] + logp[t]

    path = np.zeros(n, dtype=np.int64)
    path[-1] = int(score.argmax())

    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]

    return path

def build_domain_stats(cl_docs, it_docs):
    return {
        "CL": {
            "trans": transition_matrix(cl_docs, smooth=1.0),
            "prior": class_log_prior(cl_docs, smooth=1.0),
        },
        "IT": {
            "trans": transition_matrix(it_docs, smooth=1.0),
            "prior": class_log_prior(it_docs, smooth=1.0),
        },
    }

def decode_docs(logits, meta, domain_stats, alpha, beta):
    # logits aligned with meta=(doc_id, sent_idx, domain)
    by_doc = {}
    for i, (did, si, dom) in enumerate(meta):
        by_doc.setdefault(did, []).append((si, i, dom))

    pred = np.zeros(len(meta), dtype=np.int64)

    for did, items in by_doc.items():
        items.sort(key=lambda x: x[0])
        idx = [i for _, i, _ in items]
        dom = items[0][2]
        stats = domain_stats[dom]

        lp = (
            log_softmax_np(logits[idx].astype(np.float64))
            - alpha * stats["prior"][None, :]
        )

        pred[idx] = (
            viterbi_decode(lp, stats["trans"], beta)
            if beta > 0
            else lp.argmax(-1)
        )

    return pred

# Tune only on held-out DEV predictions.
dev_logits, dev_meta, dev_y = predict_logits(model, dev_loader)

domain_stats_train = build_domain_stats(
    cl_train_raw,
    it_train_raw,
)

def dev_f1(alpha, beta):
    p = decode_docs(
        dev_logits,
        dev_meta,
        domain_stats_train,
        alpha,
        beta
    )
    return _f1(
        dev_y,
        p,
        labels=list(range(NUM_LABELS)),
        average="macro",
        zero_division=0
    )

rows = []
for alpha in [-0.25, 0.0, 0.25, 0.5, 0.75]:
    for beta in [0.0, 0.25, 0.5, 1.0]:
        rows.append((dev_f1(alpha, beta), alpha, beta))

base_f1 = dev_f1(0.0, 0.0)
rows.sort(key=lambda r: (-r[0], r[2], abs(r[1])))

print(f"Baseline argmax (alpha=0, beta=0): {base_f1:.6f}")
print("Top decoding settings (dev F1, alpha, beta):")
for r in rows[:8]:
    print(f"  {r[0]:.6f}  alpha={r[1]:+.2f}  beta={r[2]:.2f}")

if rows[0][0] - base_f1 >= 0.005:
    DECODE_ALPHA, DECODE_BETA = rows[0][1], rows[0][2]
else:
    DECODE_ALPHA, DECODE_BETA = 0.0, 0.0

print("SELECTED decoding: alpha=", DECODE_ALPHA, "beta=", DECODE_BETA)

dec_ids = decode_docs(
    dev_logits,
    dev_meta,
    domain_stats_train,
    DECODE_ALPHA,
    DECODE_BETA
)

dev_pred_dec = records_to_prediction(dev_meta, dec_ids)
DEV_PRED_DEC = ROOT / "prediction_dev_decoded.json"

with open(DEV_PRED_DEC, "w", encoding="utf-8") as f:
    json.dump(dev_pred_dec, f, ensure_ascii=False, indent=2)

# Run the exact supplied evaluator again on the decoded DEV prediction.
res = subprocess.run(
    [
        sys.executable,
        str(EVAL_SCRIPT),
        "--truth-cl", str(CL_DEV_TRUTH),
        "--truth-it", str(IT_DEV_TRUTH),
        "--prediction", str(DEV_PRED_DEC),
        "--labels", str(LABELS_JSON),
    ],
    capture_output=True,
    text=True
)

print(res.stdout)
if res.stderr:
    print(res.stderr)

assert res.returncode == 0, "Decoded DEV prediction failed the exact evaluator."
print("✅ Decoded DEV prediction passed the exact Phase-3 evaluator.")

Baseline argmax (alpha=0, beta=0): 0.430887
Top decoding settings (dev F1, alpha, beta):
  0.448189  alpha=+0.25  beta=1.00
  0.448083  alpha=+0.00  beta=1.00
  0.443414  alpha=+0.25  beta=0.50
  0.441565  alpha=+0.50  beta=1.00
  0.438283  alpha=+0.00  beta=0.50
  0.437724  alpha=+0.00  beta=0.25
  0.437273  alpha=-0.25  beta=0.25
  0.436436  alpha=-0.25  beta=0.50
SELECTED decoding: alpha= 0.25 beta= 1.0
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.448189
Macro-Precision: 0.508259
Macro-Recall:    0.422978

✅ Decoded DEV prediction passed the exact Phase-3 evaluator.


In [29]:
# ============================================================
# CELL 17 — Build final train+dev records and fresh Phase-1+LoRA model
# ============================================================

BEST_EPOCH = int(best_obj["best_epoch"])
assert BEST_EPOCH >= 1
print("Final refit epochs:", BEST_EPOCH)

all_labeled_records = (
    make_labeled_records(cl_train_raw, "CL")
    + make_labeled_records(it_train_raw, "IT")
    + make_labeled_records(cl_dev_raw, "CL")
    + make_labeled_records(it_dev_raw, "IT")
)

all_counts = Counter(x["label"] for x in all_labeled_records)
N_all = len(all_labeled_records)

raw_w_all = np.array(
    [math.sqrt(N_all / (NUM_LABELS * max(all_counts[lab], 1))) for lab in LABELS],
    dtype=np.float32
)
raw_w_all = raw_w_all / raw_w_all.mean()
class_w_all = np.clip(raw_w_all, CLASS_WEIGHT_MIN, CLASS_WEIGHT_MAX)
class_w_all = torch.tensor(class_w_all, dtype=torch.float32, device=DEVICE)

def build_fresh_rr_from_phase1():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    bb = build_backbone()
    obj = torch.load(PHASE1_CKPT, map_location="cpu", weights_only=False)
    st = obj["model"] if isinstance(obj, dict) and "model" in obj else obj
    bb.load_state_dict(st, strict=True)

    inject_qv_lora(
        bb,
        r=LORA_R,
        alpha=LORA_ALPHA,
        dropout=LORA_DROPOUT
    )

    m = RRClassifier(bb, NUM_LABELS, HEAD_DROPOUT)
    for n_, p in m.named_parameters():
        if n_.startswith(HEAD_PREFIXES):
            p.requires_grad = True
    return m.to(DEVICE)

final_model = build_fresh_rr_from_phase1()

all_loader = DataLoader(
    RRDataset(all_labeled_records, labeled=True),
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_rr,
    num_workers=2,
    pin_memory=True,
)

final_lora_params = []
final_head_params = []
for name, p in final_model.named_parameters():
    if not p.requires_grad:
        continue
    if "lora_A" in name or "lora_B" in name:
        final_lora_params.append(p)
    else:
        final_head_params.append(p)

final_optimizer = torch.optim.AdamW(
    [
        {"params": final_lora_params, "lr": LORA_LR},
        {"params": final_head_params, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

final_updates_per_epoch = math.ceil(len(all_loader) / GRAD_ACCUM_STEPS)
final_total_updates = BEST_EPOCH * final_updates_per_epoch
final_warmup = max(1, int(final_total_updates * WARMUP_RATIO))

def final_lr_mult(step):
    if step < final_warmup:
        return max(1e-8, step / final_warmup)
    p = (step-final_warmup) / max(1, final_total_updates-final_warmup)
    return 0.5 * (1 + math.cos(math.pi * min(max(p, 0.0), 1.0)))

final_scheduler = torch.optim.lr_scheduler.LambdaLR(
    final_optimizer,
    lr_lambda=final_lr_mult
)
final_criterion = nn.CrossEntropyLoss(weight=class_w_all)
final_scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

print("All labeled records:", len(all_labeled_records))
print("Final updates:", final_total_updates)

Final refit epochs: 3
All labeled records: 18987
Final updates: 3561


In [30]:
# ============================================================
# CELL 18 — Final refit on train+dev for BEST_EPOCH
# ============================================================

final_optimizer.zero_grad(set_to_none=True)
global_update = 0

for epoch in range(1, BEST_EPOCH + 1):
    final_model.train()
    running = 0.0
    n_batches = 0
    t0 = time.time()

    for step, batch in enumerate(all_loader, start=1):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        key_valid = batch["key_valid"].to(DEVICE, non_blocking=True)
        lengths = batch["lengths"].to(DEVICE, non_blocking=True)
        pos = batch["pos"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            logits = final_model(input_ids, key_valid, lengths, pos)
            loss = final_criterion(logits, labels)
            scaled_loss = loss / GRAD_ACCUM_STEPS

        if not torch.isfinite(loss.detach()):
            raise FloatingPointError(f"Non-finite refit loss at epoch={epoch}, batch={step}")
        final_scaler.scale(scaled_loss).backward()

        running += float(loss.detach().cpu())
        n_batches += 1

        if step % GRAD_ACCUM_STEPS == 0 or step == len(all_loader):
            final_scaler.unscale_(final_optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in final_model.parameters() if p.requires_grad],
                GRAD_CLIP
            )
            final_scaler.step(final_optimizer)
            final_scaler.update()
            final_optimizer.zero_grad(set_to_none=True)
            final_scheduler.step()
            global_update += 1

        if step % 300 == 0 or step == len(all_loader):
            print(
                f"refit epoch={epoch}/{BEST_EPOCH} "
                f"batch={step}/{len(all_loader)} "
                f"loss={running/max(n_batches,1):.4f}",
                flush=True
            )

    print(
        f"Refit epoch {epoch} complete | "
        f"loss={running/max(n_batches,1):.4f} | "
        f"elapsed={time.time()-t0:.1f}s"
    )

final_task_state = {
    k: v.detach().cpu()
    for k, v in final_model.state_dict().items()
    if (
        "lora_A" in k
        or "lora_B" in k
        or k.startswith(HEAD_PREFIXES)
    )
}

torch.save({
    "task_state": final_task_state,
    "refit_epochs": BEST_EPOCH,
    "source_best_dev_f1": best_f1,
    "labels": LABELS,
    "label_to_id": LABEL_TO_ID,
}, FINAL_CKPT)

print("✅ Final refit saved:", FINAL_CKPT)

refit epoch=1/3 batch=300/4747 loss=2.1089
refit epoch=1/3 batch=600/4747 loss=1.8776
refit epoch=1/3 batch=900/4747 loss=1.7465
refit epoch=1/3 batch=1200/4747 loss=1.6316
refit epoch=1/3 batch=1500/4747 loss=1.5609
refit epoch=1/3 batch=1800/4747 loss=1.4962
refit epoch=1/3 batch=2100/4747 loss=1.4333
refit epoch=1/3 batch=2400/4747 loss=1.3929
refit epoch=1/3 batch=2700/4747 loss=1.3512
refit epoch=1/3 batch=3000/4747 loss=1.3128
refit epoch=1/3 batch=3300/4747 loss=1.2743
refit epoch=1/3 batch=3600/4747 loss=1.2435
refit epoch=1/3 batch=3900/4747 loss=1.2126
refit epoch=1/3 batch=4200/4747 loss=1.1847
refit epoch=1/3 batch=4500/4747 loss=1.1603
refit epoch=1/3 batch=4747/4747 loss=1.1410
Refit epoch 1 complete | loss=1.1410 | elapsed=685.9s
refit epoch=2/3 batch=300/4747 loss=0.5929
refit epoch=2/3 batch=600/4747 loss=0.5880
refit epoch=2/3 batch=900/4747 loss=0.5769
refit epoch=2/3 batch=1200/4747 loss=0.5769
refit epoch=2/3 batch=1500/4747 loss=0.5648
refit epoch=2/3 batch=1800/4

In [31]:
# ============================================================
# CELL 19 — Build test sentence records
# ============================================================

def make_test_records(test_dict, domain):
    records = []
    for raw_id, sents in test_dict.items():
        did = normalize_id(raw_id)
        assert isinstance(sents, list)
        for i, sent in enumerate(sents):
            records.append({
                "doc_id": did,
                "sent_idx": i,
                "domain": domain,
                "prev": " ".join(sents[max(0, i-CONTEXT_SENTS):i]) if CONTEXT_SENTS > 0 else "",
                "target": sent,
                "next": " ".join(sents[i+1:i+1+CONTEXT_SENTS]) if CONTEXT_SENTS > 0 else "",
                "pos": position_features(i, len(sents)),
            })
    return records

test_records = (
    make_test_records(cl_test_raw, "CL")
    + make_test_records(it_test_raw, "IT")
)

test_loader = DataLoader(
    RRDataset(test_records, labeled=False),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_rr,
    num_workers=2,
    pin_memory=True,
)

print("Total test sentence records:", len(test_records))
assert len(test_records) == (
    sum(len(v) for v in cl_test_raw.values())
    + sum(len(v) for v in it_test_raw.values())
)

Total test sentence records: 2197


In [32]:
# ============================================================
# CELL 20 — Test inference with transition-aware decoding
# `model` = best held-out checkpoint (train only); `final_model` = train+dev refit.
# ============================================================

@torch.no_grad()
def predict_test_logits(m, loader):
    m.eval()
    outs, meta = [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        key_valid = batch["key_valid"].to(DEVICE, non_blocking=True)
        lengths = batch["lengths"].to(DEVICE, non_blocking=True)
        pos = batch["pos"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            lg = m(input_ids, key_valid, lengths, pos)
        outs.append(lg.float().cpu().numpy())
        meta.extend(zip(batch["doc_id"], batch["sent_idx"], batch["domain"]))
    return np.concatenate(outs, 0), meta

domain_stats_all = build_domain_stats(
    cl_train_raw + cl_dev_raw,
    it_train_raw + it_dev_raw,
)

def build_prediction(m, name):
    lg, meta = predict_test_logits(m, test_loader)
    ids = decode_docs(lg, meta, domain_stats_all, DECODE_ALPHA, DECODE_BETA)
    pred = records_to_prediction(meta, ids)
    cnt = Counter(ids.tolist())
    print(f"[{name}] predicted label counts:", {LABELS[k]: cnt[k] for k in sorted(cnt)})
    return pred

predictions = {
    "heldout": build_prediction(model, "heldout"),
    "refit":   build_prediction(final_model, "refit"),
}
prediction = predictions["refit"]          # primary submission.zip

with open(OFFICIAL_PRED, "w", encoding="utf-8") as f:
    json.dump(prediction, f, ensure_ascii=False, indent=2)

print("Saved:", OFFICIAL_PRED)
print("Prediction documents:", len(prediction))


[heldout] predicted label counts: {'Fact': 483, 'Issue': 7, 'ArgumentPetitioner': 282, 'ArgumentRespondent': 72, 'Statute': 199, 'Dissent': 13, 'PrecedentReliedUpon': 385, 'PrecedentNotReliedUpon': 29, 'RulingByLowerCourt': 110, 'RatioOfTheDecision': 554, 'RulingByPresentCourt': 44, 'None': 19}
[refit] predicted label counts: {'Fact': 473, 'Issue': 15, 'ArgumentPetitioner': 235, 'ArgumentRespondent': 74, 'Statute': 182, 'Dissent': 3, 'PrecedentReliedUpon': 333, 'PrecedentNotReliedUpon': 15, 'RulingByLowerCourt': 132, 'RatioOfTheDecision': 676, 'RulingByPresentCourt': 36, 'None': 23}
Saved: /kaggle/working/phase3_exp1_model4_lora/prediction.json
Prediction documents: 10


In [33]:
# ============================================================
# CELL 21 — Strict submission validation + ZIP
# Mirrors evaluator structural rules for private test data.
# ============================================================

def normalized_test_map(raw):
    out = {}
    for raw_id, sents in raw.items():
        did = normalize_id(raw_id)
        assert did not in out, f"Duplicate normalized ID: {did}"
        out[did] = sents
    return out

cl_test = normalized_test_map(cl_test_raw)
it_test = normalized_test_map(it_test_raw)

assert not (set(cl_test) & set(it_test)), "CL/IT test IDs overlap after normalization."

expected = {}
expected.update(cl_test)
expected.update(it_test)

assert set(prediction) == set(expected), (
    f"ID mismatch: missing={set(expected)-set(prediction)}, "
    f"extra={set(prediction)-set(expected)}"
)

allowed = set(LABELS)
for doc_id, sents in expected.items():
    pred = prediction[doc_id]
    assert isinstance(pred, list), doc_id
    assert len(pred) == len(sents), (
        doc_id, len(pred), len(sents)
    )
    assert all(x in allowed for x in pred), doc_id

with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "w",
    zipfile.ZIP_DEFLATED
) as z:
    z.write(OFFICIAL_PRED, arcname="prediction.json")

with zipfile.ZipFile(SUBMISSION_ZIP, "r") as z:
    names = z.namelist()
    assert names == ["prediction.json"], names
    zipped = json.loads(z.read("prediction.json").decode("utf-8"))

assert zipped == prediction

print("✅ FINAL SUBMISSION PASSED STRUCTURAL CHECKS")
print("Documents:", len(prediction))
print("Sentences:", sum(len(v) for v in prediction.values()))
print("ZIP contents:", names)
print("prediction.json:", OFFICIAL_PRED)
print("submission.zip :", SUBMISSION_ZIP)

# ---- also write the two variants separately (upload both; keep the better)
def write_variant(name, pred):
    assert set(pred) == set(expected)
    for doc_id, sents in expected.items():
        assert len(pred[doc_id]) == len(sents), (name, doc_id)
        assert all(x in allowed for x in pred[doc_id]), (name, doc_id)
    jp = ROOT / f"prediction_{name}.json"
    zp = ROOT / f"submission_{name}.zip"
    with open(jp, "w", encoding="utf-8") as f:
        json.dump(pred, f, ensure_ascii=False)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    print("✅", zp.name)

for name, pred in predictions.items():
    write_variant(name, pred)
print("Upload BOTH submission_heldout.zip and submission_refit.zip; keep whichever Codabench scores higher.")


# Evaluator-style structural contract check on TEST.
# Test gold labels are private, so only IDs, lengths and allowed labels can be verified.
def test_submission_contract(prediction_obj):
    if not isinstance(prediction_obj, dict):
        raise ValueError(
            "prediction.json must be a JSON object mapping document IDs to label lists."
        )

    normalized = {}

    for raw_id, labels in prediction_obj.items():
        did = normalize_id(raw_id)

        if did in normalized:
            raise ValueError(
                f"Duplicate document ID after .txt normalization: {did}"
            )

        normalized[did] = labels

    if set(normalized) != set(expected):
        missing = sorted(set(expected) - set(normalized))
        extra = sorted(set(normalized) - set(expected))
        raise ValueError(
            f"ID mismatch: missing={len(missing)}, extra={len(extra)}"
        )

    for did, source_sentences in expected.items():
        pred = normalized[did]

        if not isinstance(pred, list):
            raise ValueError(f"{did}: labels must be a JSON list.")

        if len(pred) != len(source_sentences):
            raise ValueError(
                f"{did}: sentence-count mismatch "
                f"({len(pred)} vs {len(source_sentences)})."
            )

        if any(x not in allowed for x in pred):
            raise ValueError(f"{did}: unknown label encountered.")

    return True

assert test_submission_contract(prediction)
print(
    "✅ TEST prediction satisfies the same ID / sentence-count / "
    "allowed-label contract enforced by the Phase-3 evaluator."
)


✅ FINAL SUBMISSION PASSED STRUCTURAL CHECKS
Documents: 10
Sentences: 2197
ZIP contents: ['prediction.json']
prediction.json: /kaggle/working/phase3_exp1_model4_lora/prediction.json
submission.zip : /kaggle/working/phase3_exp1_model4_lora/submission.zip
✅ submission_heldout.zip
✅ submission_refit.zip
Upload BOTH submission_heldout.zip and submission_refit.zip; keep whichever Codabench scores higher.
✅ TEST prediction satisfies the same ID / sentence-count / allowed-label contract enforced by the Phase-3 evaluator.


## What to report for Experiment 1

Record these after the run:

- best dev epoch
- combined CL+IT Macro-F1 / Macro-Precision / Macro-Recall from the **exact supplied evaluator**
- CL-only and IT-only diagnostic Macro-F1
- per-label F1
- trainable parameter count / percentage
- final Codabench score
- runtime and GPU

For the final report, describe this as **custom LoRA implemented from scratch on Model 4 Q/V projections**, with the Phase-1 backbone frozen and a 13-class sentence-level classification head.
- decoding setting chosen (alpha, beta) and dev F1 with vs without it
- ablations for the required analysis: `CONTEXT_SENTS=0` (sentence-only), `USE_POS=False`, then compare per-label F1 and CL vs IT
- Codabench score of heldout vs refit


### v3 safeguards

- The exact supplied evaluator is executed on the held-out combined CL+IT prediction before final refit.
- The decoded DEV prediction is independently sent through the same evaluator.
- Test output receives evaluator-style structural checks because private test labels cannot be scored locally.
- CL and IT transition statistics are estimated separately with smoothing; no label is hard-forbidden.
- Final refit starts fresh from the same Phase-1 checkpoint with a deterministic seed.
